In [1]:
from typing import List, TypedDict
import time

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate

from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv

load_dotenv()

d:\Coding\Genarative-AI\myvenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
from langchain_huggingface import ChatHuggingFace,HuggingFaceEndpoint

In [3]:
from langchain_huggingface import HuggingFaceEmbeddings

In [4]:
docs = (
    PyPDFLoader('D:/Coding/Genarative-AI/documents/book1.pdf').load() +
    PyPDFLoader('D:/Coding/Genarative-AI/documents/book2.pdf').load() +
    PyPDFLoader('D:/Coding/Genarative-AI/documents/book3.pdf').load()
)

type(docs)
print(len(docs))

2123


In [5]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=175)

chunks = text_splitter.split_documents(docs)


for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")

In [6]:
print(len(chunks))

7456


In [7]:
embedding = HuggingFaceEmbeddings(
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6250.19it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [8]:
vector_store = FAISS.from_documents(chunks,embedding)

In [9]:
retriever = vector_store.as_retriever(search_type = 'similarity', search_kwarg = {'k':4})

In [10]:
llm = HuggingFaceEndpoint(
    repo_id = 'zai-org/GLM-4.6',
    task = 'text-generation'
)

In [12]:
class state(TypedDict):
    question = str
    docs = list[Document]
    answer = str

In [13]:
def retrive(state):
    q = state['question']
    return {'docs': retriever.invoke(q)}

In [14]:
import os
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

# ১. ভেক্টর ডাটাবেস সেভ করার কোড:
# আপনার নোটবুকে যেখানে vectorstore (বা আপনার দেওয়া অন্য কোনো নাম) তৈরি হয়েছে, 
# তার ঠিক নিচে এই লাইনগুলো লিখে রান করবেন:

def save_my_vector_db(vectorstore, folder_name="my_faiss_index"):
    vectorstore.save_local(folder_name)
    print(f"ভেক্টর ডাটাবেস সফলভাবে '{folder_name}' ফোল্ডারে সেভ হয়েছে!")

# উদাহরণ: 
# save_my_vector_db(vectorstore)


# ২. পরবর্তীতে ডাটাবেস লোড করার কোড:
# নতুন করে আর পিডিএফ লোড বা চ্যাঙ্ক না করে সরাসরি সেভ করা ডাটাবেস লোড করুন:

def load_my_vector_db(folder_name="my_faiss_index"):
    # আপনার যেই এমবেডিং মডেল ব্যবহার করা হয়েছিল সেটি ডিফাইন করুন
    embeddings = OpenAIEmbeddings() 
    
    # allow_dangerous_deserialization=True দেওয়াটা আবশ্যক
    loaded_vectorstore = FAISS.load_local(
        folder_name, 
        embeddings, 
        allow_dangerous_deserialization=True
    )
    print(f"'{folder_name}' ফোল্ডার থেকে ভেক্টর ডাটাবেস সফলভাবে লোড হয়েছে!")
    return loaded_vectorstore

# উদাহরণ:
# new_vectorstore = load_my_vector_db()
